# Web and social media analysis
A project of social media analysis about these questions:

1. **Sentiment & framing** — Do people talk about MT as a tool, a threat, or both? How does framing differ between users (translators vs. general public vs. tech advocates)?
2. **Professional impact discourse** — How do working translators and localizers discuss MT's effect on their careers, rates, and job availability?
3. **Utility vs. displacement tension** — Do people acknowledge MT's utility while also lamenting professional erosion, or are these largely separate conversations?

## Setup

In [1]:
!pip -q install pandas atproto python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
import time
import re
from datetime import datetime, timezone, timedelta
import pandas as pd

from atproto import Client, models
from atproto_client.exceptions import RequestException, InvokeTimeoutError, NetworkError
from dotenv import load_dotenv
import os

## Data gathering

Use BlueSky social media, their API to fetch posts, threads, users and crawl to get connections.

In [3]:
# SearchPostsResponse
# │
# ├── posts: List[PostView]
# │   │
# │   └── PostView
# │       ├── uri          # unique ID of the post (e.g. at://did:plc:.../post/abc123)
# │       ├── cid          # content hash
# │       ├── like_count
# │       ├── repost_count
# │       ├── reply_count
# │       │
# │       ├── author: ProfileViewBasic
# │       │   ├── handle       # e.g. "janedoe.bsky.social"
# │       │   ├── display_name
# │       │   └── description  # ← this is the bio, useful for user labeling
# │       │
# │       └── record: Record   # ← the actual post content
# │           ├── text         # ← the post text you want
# │           ├── created_at   # timestamp
# │           └── reply: Optional
# │               ├── parent.uri   # uri of the post this replies to
# │               └── root.uri     # uri of the thread root
# │
# └── cursor   # pagination token — pass this to get the next page of results

Data gathering pipeline:

1. Search keywords → seed posts
2. Crawl threads of each seed post → more posts + reply structure (needed for graph!)
3. Crawl following of top authors → more users
4. Deduplicate everything → final dataset

In [4]:
from atproto import Client

load_dotenv()
client = Client()

client.login(os.getenv("BSKY_HANDLE"), os.getenv("BSKY_PASSWORD"))

ProfileViewDetailed(did='did:plc:azbnjczlamokwskrcuwdks7m', handle='linda-sta.bsky.social', associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=None, feedgens=0, germ=None, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated'), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:azbnjczlamokwskrcuwdks7m/bafkreigg7wsdyxfvak4zzxtruqa4ow45jurr4uugjpdhcpdv44dh2bsnvm', banner=None, created_at='2026-04-06T10:10:10.449Z', debug=None, description=None, display_name='', followers_count=0, follows_count=1, indexed_at='2026-04-06T10:10:30.756Z', joined_via_starter_pack=None, labels=[], pinned_post=None, posts_count=0, pronouns=None, status=None, verification=None, viewer=ViewerState(activity_subscription=None, blocked_by=False, blocking=None, blocking_by_list=None, followed_by=None, following=None, known_followers=No

In [5]:
results = client.app.bsky.feed.search_posts({'q': 'machine translation', 'lang': 'en', 'limit': 5})
post = results.posts[0]
print(post.author)  # prints the full object to explore

did='did:plc:paobv7cc3svtzqjwlsrqwaw3' handle='le0nkn1ght.bsky.social' associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=ProfileAssociatedChat(allow_incoming='following', allow_group_invites=None, py_type='app.bsky.actor.defs#profileAssociatedChat'), feedgens=None, germ=None, labeler=None, lists=None, starter_packs=None, py_type='app.bsky.actor.defs#profileAssociated') avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:paobv7cc3svtzqjwlsrqwaw3/bafkreigo2fn5x26b2ezx4os3rtgwywflhewz7g6ziukxptm3a665uef3vy' created_at='2023-10-30T17:14:55.793Z' debug=None display_name='れおん' labels=[Label(cts='1970-01-01T00:00:00.000Z', src='did:plc:paobv7cc3svtzqjwlsrqwaw3', uri='at://did:plc:paobv7cc3svtzqjwlsrqwaw3/app.bsky.actor.profile/self', val='!no-unauthenticated', cid='bafyreibnntn2kpsilm75u7s4aij3zgyxqlbigbjxsm2za3oi5f4lnx62mq', exp=None, neg=None, 

In [6]:
def call_with_retries(callable_fn, *args, max_retries: int = 10, **kwargs):
    for attempt in range(max_retries):
        try:
            return callable_fn(*args, **kwargs)
        except RequestException as e:
            resp = getattr(e, "response", None)
            status = getattr(resp, "status_code", None)
            headers = getattr(resp, "headers", {}) or {}

            if status == 429:
                reset = headers.get("ratelimit-reset") or headers.get("RateLimit-Reset")
                retry_after = headers.get("retry-after") or headers.get("Retry-After")

                if reset:
                    wait_s = max(1, int(reset) - _now_utc_ts()) + 1
                elif retry_after:
                    wait_s = int(float(retry_after))
                else:
                    wait_s = min(60, 2 ** attempt)

                print(f"[429] rate limited, sleeping {wait_s}s")
                time.sleep(wait_s)
                continue

            wait_s = min(10, 2 ** attempt)
            print(f"[{status}] request failed, sleeping {wait_s}s")
            time.sleep(wait_s)

    raise RuntimeError("Too many retries / repeated failures.")

In [7]:
# Facet extraction: hashtags, mentions, links

def extract_facets(record) -> dict:
    """Extract hashtags, mentions, links from a post record (if facets exist)."""
    hashtags = []
    mentions = []
    links = []

    facets = getattr(record, "facets", None) or []
    for facet in facets:
        features = getattr(facet, "features", None) or []
        for feat in features:
            ftype = getattr(feat, "py_type", None) or getattr(feat, "type", None)

            if ftype == "app.bsky.richtext.facet#tag":
                tag = getattr(feat, "tag", None)
                if tag:
                    hashtags.append(tag)
            elif ftype == "app.bsky.richtext.facet#mention":
                did = getattr(feat, "did", None)
                if did:
                    mentions.append({"did": did})
            elif ftype == "app.bsky.richtext.facet#link":
                uri = getattr(feat, "uri", None)
                if uri:
                    links.append(uri)

    return {"hashtags": hashtags, "mentions": mentions, "links": links}

In [8]:
# Search function with time window and facets extraction
def _parse_dt_utc(iso_str: str) -> datetime:
    dt = datetime.fromisoformat(iso_str.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)

def search_posts_time_window(
    client,
    query: str = None,
    tag: str = None,
    since_iso: str = None,
    until_iso: str = None,
    max_posts: int = 1000,
    page_size: int = 50,
    polite_sleep: float = 0.25,
    print_every_page: bool = False,
):
    cursor = None
    rows = []
    page = 0

    while len(rows) < max_posts:
        page += 1

        params = models.AppBskyFeedSearchPosts.Params(
            q=query if query else tag,
            tag=[tag] if tag else None,
            sort="latest",
            since=since_iso,
            until=until_iso,
            limit=min(page_size, max_posts - len(rows)),
            cursor=cursor,
        )

        res = call_with_retries(client.app.bsky.feed.search_posts, params)
        cursor = res.cursor
        posts = res.posts or []

        if not posts:
            if print_every_page:
                print(f"[page {page}] no posts, stopping.")
            break

        if print_every_page:
            dts = []
            for p in posts:
                created = getattr(getattr(p, 'record', None), 'created_at', None)
                if created:
                    dts.append(_parse_dt_utc(created))
            if dts:
                print(
                    f"[page {page}] newest={max(dts).isoformat()}  oldest={min(dts).isoformat()}  "
                    f"collected={len(rows)}  cursor={'yes' if cursor else 'no'}"
                )

        for p in posts:
            rec = getattr(p, "record", None)
            created = getattr(rec, "created_at", None)
            if not created:
                continue

            created_dt = _parse_dt_utc(created).strftime("%Y-%m-%dT%H:%M:%S.000Z")
            if not (since_iso <= created_dt < until_iso):
                continue

            author = getattr(p, "author", None)

            # Extract facets
            facets_data = extract_facets(rec)

            rows.append({
                "created_at": created_dt,
                "uri": p.uri,
                "cid": getattr(p, "cid", None),
                "text": getattr(rec, "text", None),
                "handle": getattr(author, "handle", None),
                "did": getattr(author, "did", None),
                "replies": getattr(p, "reply_count", None),
                "likes": getattr(p, "like_count", None),
                "reposts": getattr(p, "repost_count", None),
                "quotes": getattr(p, "quote_count", None),
                "hashtags": facets_data["hashtags"],
                "mentions": facets_data["mentions"],
                "links": facets_data["links"],
                'is_reply':     post.record.reply is not None,
                'reply_to':     post.record.reply.parent.uri if post.record.reply else None,
                'query':        query or f'#{tag}',   # track which query found it
            })

            if len(rows) >= max_posts:
                break

        if cursor is None:
            if print_every_page:
                print(f"[page {page}] cursor is None, stopping.")
            break

        time.sleep(polite_sleep)

    df = pd.DataFrame(rows)
    if not df.empty:
        df["created_at"] = pd.to_datetime(df["created_at"], utc=True, errors="coerce")
        df = df.sort_values("created_at", ascending=True).reset_index(drop=True)
    return df

In [9]:
def get_followers_list(client, actor: str, max_total: int = 300, page_size: int = 100, polite_sleep: float = 0.6):
    cursor = None
    out = []
    page = 0
    while len(out) < max_total:
        page += 1
        params = models.AppBskyGraphGetFollowers.Params(
            actor=actor,
            limit=min(page_size, max_total - len(out)),
            cursor=cursor
        )
        res = call_with_retries(client.app.bsky.graph.get_followers, params)
        cursor = res.cursor
        followers = res.followers or []

        out.extend([{
            "handle": getattr(f, "handle", None),
            "did": getattr(f, "did", None),
            "display_name": getattr(f, "display_name", None),
            "description": getattr(f, "description", None),
        } for f in followers])

        # print(f"[followers page {page}] downloaded={len(out)} cursor={'yes' if cursor else 'no'}")
        if cursor is None or not followers:
            break
        time.sleep(polite_sleep)
    return out


def get_following_list(client, actor: str, max_total: int = 300, page_size: int = 100, polite_sleep: float = 0.6):
    cursor = None
    out = []
    page = 0
    while len(out) < max_total:
        page += 1
        params = models.AppBskyGraphGetFollows.Params(
            actor=actor,
            limit=min(page_size, max_total - len(out)),
            cursor=cursor
        )
        res = call_with_retries(client.app.bsky.graph.get_follows, params)
        cursor = res.cursor
        follows = res.follows or []

        out.extend([{
            "handle": getattr(f, "handle", None),
            "did": getattr(f, "did", None),
            "display_name": getattr(f, "display_name", None),
            "description": getattr(f, "description", None),
        } for f in follows])

        # print(f"[following page {page}] downloaded={len(out)} cursor={'yes' if cursor else 'no'}")
        if cursor is None or not follows:
            break
        time.sleep(polite_sleep)
    return out

In [10]:
# Time window: last 12 months
UNTIL_ISO = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
SINCE_ISO = (datetime.now(timezone.utc) - timedelta(days=365)).strftime("%Y-%m-%dT%H:%M:%S.000Z")

print(f"Collecting from {SINCE_ISO} to {UNTIL_ISO}")

In [11]:
# Define queries and hastags to start from
TEXT_QUERIES = [
    '"machine translation"',  
    '"AI translation"',
    '"post-editing"',
    '"human translation"',
    '"translation quality"',
    '"translator jobs"',
    '"translation industry"',
    'DeepL',                   
    'MTPE',
]

COMPOUND_QUERIES = [
    '"post-editing" AI',
    '"AI translation" threat',
    'scraped translation',
    '"machine translation" slop',
    '"human translator" AI',
    '"translation industry" AI',
]

HASHTAG_QUERIES = [
    'machinetranslation',
    'MTPE',
    'localization',
    'l10n',
    'SlowTranslation',
    'HumanTranslation',
    'PauseAI',
    'NoAI',
]

In [ ]:
# API data fetching

all_dfs = []

for q in TEXT_QUERIES:
    df_q = search_posts_time_window(
        client, query=q,
        since_iso=SINCE_ISO, until_iso=UNTIL_ISO,
        max_posts=50, page_size=25, polite_sleep=0.35,
    )
    df_q['query'] = q
    all_dfs.append(df_q)

for tag in HASHTAG_QUERIES:
    df_t = search_posts_time_window(
        client, tag=tag,
        since_iso=SINCE_ISO, until_iso=UNTIL_ISO,
        max_posts=50, page_size=25, polite_sleep=0.35,
    )
    df_t['query'] = f'#{tag}'
    all_dfs.append(df_t)

for q in COMPOUND_QUERIES:
    df_q = search_posts_time_window(
        client, query=q,
        since_iso=SINCE_ISO, until_iso=UNTIL_ISO,
        max_posts=50, page_size=25, polite_sleep=0.35,
    )
    df_q['query'] = q
    all_dfs.append(df_q)

df_seeds = pd.concat(all_dfs, ignore_index=True).drop_duplicates(subset='uri').reset_index(drop=True)
print(f"\nTotal seed posts: {len(df_seeds)}")
print(f"Unique authors: {df_seeds['handle'].nunique()}")
print(f"\nBreakdown:\n{df_seeds['query'].value_counts()}")


Total seed posts: 845
Unique authors: 635

Breakdown:
query
"machine translation"         50
"human translation"           50
"translation industry"        50
DeepL                         50
MTPE                          50
#machinetranslation           50
#PauseAI                      50
#NoAI                         50
"human translator" AI         50
"AI translation"              49
"post-editing"                49
"translation quality"         49
#localization                 49
#l10n                         49
"post-editing" AI             46
"translation industry" AI     29
"machine translation" slop    25
scraped translation           21
"translator jobs"             19
#HumanTranslation              7
"AI translation" threat        3
Name: count, dtype: int64


In [13]:
print(df_seeds.columns)
df_seeds.to_csv('data/seeds.csv', index=False)
print("Saved seeds.csv")

Index(['created_at', 'uri', 'cid', 'text', 'handle', 'did', 'replies', 'likes',
       'reposts', 'quotes', 'hashtags', 'mentions', 'links', 'is_reply',
       'reply_to', 'query'],
      dtype='str')
Saved seeds.csv


In [ ]:
# Crawl the threads of seed posts

visited_uris = set(df_seeds['uri'].tolist())  # skip re-fetching the seeds
thread_posts = []
reply_edges = []

def crawl_thread(uri, depth=0, max_depth=2):
    if depth > max_depth:
        return

    try:
        thread = client.app.bsky.feed.get_post_thread({'uri': uri, 'depth': 1})
        node = thread.thread

        if not hasattr(node, 'post'):
            return

        post = node.post

        # collect this post only if not already seen
        if post.uri not in visited_uris:
            visited_uris.add(post.uri)
            facets = extract_facets(post.record)
            thread_posts.append({
                'uri':        post.uri,
                'text':       post.record.text,
                'created_at': post.record.created_at,
                'handle':     post.author.handle,
                'likes':      post.like_count,
                'reposts':    post.repost_count,
                'replies':    post.reply_count,
                'is_reply':   post.record.reply is not None,
                'reply_to':   post.record.reply.parent.uri if post.record.reply else None,
                'hashtags':   facets['hashtags'],
                'mentions':   facets['mentions'],
                'links':      facets['links'],
                'query':      'thread_crawl',
            })

            if post.record.reply:
                reply_edges.append({
                    'source': post.author.handle,
                    'target_uri': post.record.reply.parent.uri,
                    'type': 'reply',
                })

        # always recurse into replies regardless
        if hasattr(node, 'replies') and node.replies:
            for reply in node.replies:
                if hasattr(reply, 'post'):
                    crawl_thread(reply.post.uri, depth + 1, max_depth)

        time.sleep(0.3)

    except Exception as e:
        print(f"  skipped {uri}: {e}")

# only crawl seeds that have replies — no point crawling dead threads
seeds_with_replies = df_seeds[df_seeds['replies'] > 0]['uri'].tolist()
print(f"Seeds with replies: {len(seeds_with_replies)} / {len(df_seeds)}")

for i, uri in enumerate(seeds_with_replies):
    if i % 20 == 0:
        print(f"  crawling {i}/{len(seeds_with_replies)}...")
    crawl_thread(uri)

df_threads = pd.DataFrame(thread_posts).drop_duplicates(subset='uri').reset_index(drop=True)
df_reply_edges = pd.DataFrame(reply_edges)

print(f"\nNew posts from threads: {len(df_threads)}")
print(f"Reply edges found: {len(df_reply_edges)}")

Seeds with replies: 326 / 845
  crawling 0/326...
  crawling 20/326...
  crawling 40/326...
  crawling 60/326...
  crawling 80/326...
  crawling 100/326...
  crawling 120/326...
  crawling 140/326...
  crawling 160/326...
  crawling 180/326...
  crawling 200/326...
  crawling 220/326...
  crawling 240/326...
  crawling 260/326...
  crawling 280/326...
  crawling 300/326...
  crawling 320/326...

New posts from threads: 696
Reply edges found: 696


In [15]:
df_seeds.head()

,created_at,uri,cid,text,handle,did,replies,likes,reposts,quotes,hashtags,mentions,links,is_reply,reply_to,query
0,2026-06-19 12:51:48+00:00,at://did:plc:fmfmolqd4gzgvgt5bmtgkbf6/app.bsky...,bafyreigoryxm2fkivrkn62euthekiafblxozqol3p2ha7...,it's crazy that they're depending on machine t...,ukeblender.bsky.social,did:plc:fmfmolqd4gzgvgt5bmtgkbf6,1.0,2.0,0.0,0.0,[],[],[],True,at://did:plc:ico7mhqftrjhdgqm73ejsptd/app.bsky...,"""machine translation"""
1,2026-06-19 13:55:57+00:00,at://did:plc:onurlv66npmwhguefdfgtrap/app.bsky...,bafyreibe6ga6hht5daaa4eidtrpaf5mo6nhpr4jibytre...,​​Bandai literally embodying Oldtype mentality...,zeonicscans.bsky.social,did:plc:onurlv66npmwhguefdfgtrap,3.0,79.0,26.0,1.0,[],[],[],True,at://did:plc:ico7mhqftrjhdgqm73ejsptd/app.bsky...,"""machine translation"""
2,2026-06-19 14:01:12+00:00,at://did:plc:sd5j22tuaciqcezjcl5wym7p/app.bsky...,bafyreier4t5yzqrqg5f7vyn6qddehsffcrex2wcquthyc...,[Skeb] クライアントOCのKoshoさん 戦場おちんぽ漁りデカパイズリ\nwww.pi...,out-tome.bsky.social,did:plc:sd5j22tuaciqcezjcl5wym7p,1.0,10.0,7.0,0.0,[],[{'did': 'did:plc:syeadck5h4wh4m6nt6wsw4c6'}],[https://www.pixiv.net/artworks/146203193],True,at://did:plc:ico7mhqftrjhdgqm73ejsptd/app.bsky...,"""machine translation"""
3,2026-06-19 14:07:43+00:00,at://did:plc:sd5j22tuaciqcezjcl5wym7p/app.bsky...,bafyreied2sqz54454cvjtfkn2bnxf3h5l7vapzegutfvr...,リンクが切れていたので再投稿しました。\n削除前のポストにいいねやリポスト、ブックマークして...,out-tome.bsky.social,did:plc:sd5j22tuaciqcezjcl5wym7p,0.0,4.0,2.0,0.0,[],[],[https://skeb.jp/@OutTome2],True,at://did:plc:ico7mhqftrjhdgqm73ejsptd/app.bsky...,"""machine translation"""
4,2026-06-19 15:15:10+00:00,at://did:plc:xmjeqqdhsfz7p2lsrrlss7ef/app.bsky...,bafyreibfwrlj4x5wzmsiwryjydafc5yw2s54yyevc7l7a...,I still rely pretty heavily on dictionaries an...,ninagonzalez.bsky.social,did:plc:xmjeqqdhsfz7p2lsrrlss7ef,1.0,1.0,0.0,0.0,[],[],[],True,at://did:plc:ico7mhqftrjhdgqm73ejsptd/app.bsky...,"""machine translation"""


In [ ]:
# Consistency check - make sure seeds also have the same columns
df_seeds['is_reply'] = df_seeds['reply_to'].notna()

df_all_posts = pd.concat([df_seeds, df_threads], ignore_index=True)
df_all_posts = df_all_posts.drop_duplicates(subset='uri').reset_index(drop=True)

print(f"Total posts: {len(df_all_posts)}")
print(f"Unique authors: {df_all_posts['handle'].nunique()}")

df_all_posts.to_csv('data/posts_raw.csv', index=False)
df_reply_edges.to_csv('data/reply_edges.csv', index=False)
print("Saved posts_raw.csv and reply_edges.csv")

Total posts: 1541
Unique authors: 942
Saved posts_raw.csv and reply_edges.csv


In [38]:
# Only crawl the followings of the most active users, not all
author_counts = df_all_posts['handle'].value_counts()
top_authors = author_counts[author_counts > 2].index.tolist()
print(f"Authors with more than 2 post: {len(top_authors)}")

Authors with more than 2 post: 127


In [ ]:
# ── AUTHOR/FOLLOWER CRAWL ──────────────────────────────
# Crawl the following of the top authors

follow_edges = []
author_profiles = []

for i, handle in enumerate(top_authors):

    try:
        # get bio
        profile = client.app.bsky.actor.get_profile({'actor': handle})
        author_profiles.append({
            'handle':       handle,
            'display_name': profile.display_name,
            'bio':          profile.description,
            'followers':    profile.followers_count,
            'following':    profile.follows_count,
            'posts_count':  profile.posts_count,
        })
        time.sleep(0.3)

        # get followers (capped at 100 per author)
        followers = get_followers_list(client, handle, max_total=100, page_size=100, polite_sleep=0.3)
        for f in followers:
            follow_edges.append({
                'source': f['handle'],
                'target': handle,
                'type':   'follows',
            })
        time.sleep(0.3)
        if i % 100 == 0:
            print(f"  crawling {i}/{len(top_authors)}...")
    except Exception as e:
        print(f"  skipped {handle}: {e}")

df_profiles = pd.DataFrame(author_profiles)
df_follow_edges = pd.DataFrame(follow_edges)

print(f"\nProfiles collected: {len(df_profiles)}")
print(f"Follow edges collected: {len(df_follow_edges)}")

df_profiles.to_csv('data/author_profiles.csv', index=False)
df_follow_edges.to_csv('data/follow_edges.csv', index=False)
print("Saved author_profiles.csv and follow_edges.csv")


Profiles collected: 127
Follow edges collected: 10219
Saved author_profiles.csv and follow_edges.csv


In [19]:
print(df_profiles.head())
print(df_profiles['handle'].value_counts().head())

                       handle        display_name  \
0      babygoldie.bsky.social  News by babygoldie   
1                 liabelle.me         Lia Belle 🌷   
2         pauseai.bsky.social             PauseAI   
3        eamt2026.bsky.social                       
4  json-translate.bsky.social      JSON Translate   

                                                 bio  followers  following  \
0  Posts via automated bot. Likes and comments by...        433          2   
1  Blogger. Content Marketer. News Junkie. Avid R...        491         37   
2  Community of volunteers who work together to m...        851         35   
3                                                NaN         23         14   
4               Easily translate JSON language files         21         83   

   posts_count  
0       394773  
1         1204  
2          353  
3           54  
4          113  
handle
babygoldie.bsky.social        1
liabelle.me                   1
pauseai.bsky.social           1
eamt202

In [ ]:
# ── COMBINE EDGES ────────────────────────────────────────────────

# reply edges need target handle, not uri — resolve later
# for now just label them differently and combine
df_reply_edges['source'] = df_reply_edges['source']
df_reply_edges['target'] = df_reply_edges['target_uri']  # uri for now
df_reply_edges = df_reply_edges[['source', 'target', 'type']]

df_graph = pd.concat([df_follow_edges, df_reply_edges], ignore_index=True)

print(f"Total graph edges: {len(df_graph)}")
print(f"  follow edges: {len(df_follow_edges)}")
print(f"  reply edges:  {len(df_reply_edges)}")

df_graph.to_csv('data/graph.csv', index=False)
print("Saved graph.csv")

Total graph edges: 10915
  follow edges: 10219
  reply edges:  696
Saved graph.csv


## Preprocessing
Deduplicating data, cleaning, removing non-english posts or posts that are not connected to the chosen topic.

In [ ]:
# RUN FROM HERE ON SAVED RAW DATA - not calling API anymore, now local cleaning
df = pd.read_csv('data/posts_raw.csv')
df_profiles = pd.read_csv('data/author_profiles.csv')
df_reply_edges = pd.read_csv('data/reply_edges.csv')
df_follow_edges = pd.read_csv('data/follow_edges.csv')

print(f"Posts loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")

Posts loaded: 1541
Columns: ['created_at', 'uri', 'cid', 'text', 'handle', 'did', 'replies', 'likes', 'reposts', 'quotes', 'hashtags', 'mentions', 'links', 'is_reply', 'reply_to', 'query']


In [22]:
before = len(df)
df = df.drop_duplicates(subset='uri').reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicates, {len(df)} remaining")

Dropped 0 duplicates, 1541 remaining


In [23]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)       # remove URLs
    text = re.sub(r'\s+', ' ', text).strip()  # normalize whitespace
    return text

df['text_clean'] = df['text'].apply(clean_text)

In [24]:
df = df[df['text_clean'].str.len() > 10].reset_index(drop=True)
print(f"Posts after empty drop: {len(df)}")

Posts after empty drop: 1496


In [25]:
# Topical relevance filtering
off_topic_queries = ['#PauseAI', '#NoAI', '#localization']

for q in off_topic_queries:
    print(f"\n── {q} ──")
    sample = df[df['query'] == q]['text_clean'].sample(min(5, len(df[df['query'] == q]))).tolist()
    for t in sample:
        print(f"  • {t[:120]}")


── #PauseAI ──
  • We need to PauseAI now, before risks like bioweapons cause a catastrophe!
  • Now that's fucking crazy .... AI is given way too much power. There is no intelligence in AI. People need to realize thi
  • Leading AI scientist, professor Stuart Russell, expresses concern over a lack of understanding of how AI models work. #p
  • - Take action now to help save literally everyone on Earth by getting involved with groups like @pauseai @StopAI_Info @J
  • PauseAI wants to hear from you! Is AI affecting your work, education, security, creativity or daily life? If so, tell us

── #NoAI ──
  • Caduceus 💜 Had to start this one over from scratch twice but I think it paid off~ #criticalrolefanart #criticalrolecampa
  • Neofeud 2, my #cyberpunk 90's-style game where an ex-robo-marine and a rebel hacker struggle against an empire on its la
  • "Kodachrome Meltdown: House Of God" Order prints at samuelxl5.darkroom.com/products/the... #photography #art #filmphotog
  • Icarus Day. He

In [26]:
MT_TERMS = [
    'translat', 'interpret', 'locali', 'localiz',  # root forms catch variants
    'deepl', 'mtpe', 'post-edit', 'post edit',
    'language model', 'nlp', 'multilingual',
    'linguist', 'terminology', 'glossary',
]

def is_relevant(text):
    text_lower = text.lower()
    return any(term in text_lower for term in MT_TERMS)

before = len(df)
df = df[df['text_clean'].apply(is_relevant)].reset_index(drop=True)
print(f"Dropped {before - len(df)} irrelevant posts")
print(f"Remaining: {len(df)}")
print(f"\nBreakdown by query:")
print(df['query'].value_counts())

Dropped 638 irrelevant posts
Remaining: 858

Breakdown by query:
query
thread_crawl                  164
"translation industry"         50
DeepL                          50
#machinetranslation            50
"human translation"            48
"translation quality"          48
#localization                  48
"AI translation"               47
"post-editing"                 47
MTPE                           47
"machine translation"          46
"post-editing" AI              45
"human translator" AI          42
"translation industry" AI      29
#l10n                          27
"machine translation" slop     23
"translator jobs"              19
scraped translation            17
#HumanTranslation               7
"AI translation" threat         3
#NoAI                           1
Name: count, dtype: int64


In [27]:
# User type labeling

PROFESSIONAL_TERMS = [
    'translat', 'interpret', 'locali', 'linguist', 
    'terminolog', 'lexicograph', 'subtitl', 'transcrib'
]

TECH_TERMS = [
    'developer', 'engineer', 'nlp', 'machine learning', 'ai researcher',
    'data scientist', 'software', 'programmer', 'coder', 'researcher'
]

def label_user(bio):
    if not isinstance(bio, str):
        return 'general'
    bio = bio.lower()
    if any(t in bio for t in PROFESSIONAL_TERMS):
        return 'professional'
    if any(t in bio for t in TECH_TERMS):
        return 'tech'
    return 'general'

bio_map = dict(zip(df_profiles['handle'], df_profiles['bio']))
df['bio'] = df['handle'].map(bio_map)
df['user_type'] = df['bio'].apply(label_user)

print(df['user_type'].value_counts())

user_type
general         752
professional     87
tech             19
Name: count, dtype: int64


In [28]:
df_profiles.head()

,handle,display_name,bio,followers,following,posts_count
0,babygoldie.bsky.social,News by babygoldie,Posts via automated bot. Likes and comments by...,433,2,394773
1,liabelle.me,Lia Belle 🌷,Blogger. Content Marketer. News Junkie. Avid R...,491,37,1204
2,pauseai.bsky.social,PauseAI,Community of volunteers who work together to m...,851,35,353
3,eamt2026.bsky.social,NaN,NaN,23,14,54
4,json-translate.bsky.social,JSON Translate,Easily translate JSON language files,21,83,113


In [ ]:
# resolve reply edges uri → handle
uri_to_handle = dict(zip(df['uri'], df['handle']))
df_reply_edges['target'] = df_reply_edges['target_uri'].map(uri_to_handle)
df_reply_edges = df_reply_edges.dropna(subset=['target'])  # drop unresolvable
df_reply_edges = df_reply_edges[['source', 'target', 'type']]

# rebuild graph.csv with resolved handles
df_graph = pd.concat([df_follow_edges, df_reply_edges], ignore_index=True)
print(f"Reply edges resolved: {len(df_reply_edges)}")
print(f"Total graph edges: {len(df_graph)}")

# save clean datasets
df_graph.to_csv('data/graph.csv', index=False)

print("\nSaved:")
print(f"  graph.csv        — {len(df_graph)} edges")

Reply edges resolved: 443
Total graph edges: 10662

Saved:
  posts_clean.csv  — 858 posts
  graph.csv        — 10662 edges
  author_profiles.csv — 127 profiles


In [30]:
# ── CONSISTENCY CHECKS ───────────────────────────────────────────

print("═" * 50)
print("POSTS CLEAN")
print("═" * 50)
print(f"Total posts:        {len(df)}")
print(f"Unique URIs:        {df['uri'].nunique()}  (should match total)")
print(f"Unique authors:     {df['handle'].nunique()}")
print(f"Null text:          {df['text_clean'].isnull().sum()}  (should be 0)")
print(f"Null handle:        {df['handle'].isnull().sum()}  (should be 0)")
print(f"Null created_at:    {df['created_at'].isnull().sum()}  (should be 0)")
print(f"Date range:         {df['created_at'].min()} → {df['created_at'].max()}")
print(f"User types:         {df['user_type'].value_counts().to_dict()}")

print("\n" + "═" * 50)
print("GRAPH")
print("═" * 50)
print(f"Total edges:        {len(df_graph)}")
print(f"Edge types:         {df_graph['type'].value_counts().to_dict()}")
print(f"Null source:        {df_graph['source'].isnull().sum()}  (should be 0)")
print(f"Null target:        {df_graph['target'].isnull().sum()}  (should be 0)")
print(f"Unique sources:     {df_graph['source'].nunique()}")
print(f"Unique targets:     {df_graph['target'].nunique()}")

print("\n" + "═" * 50)
print("PROFILES")
print("═" * 50)
print(f"Total profiles:     {len(df_profiles)}")
print(f"Null bios:          {df_profiles['bio'].isnull().sum()}")
print(f"All in posts:       {df_profiles['handle'].isin(df['handle']).all()}  (should be True)")

print("\n" + "═" * 50)
print("CROSS-FILE")
print("═" * 50)
graph_handles = set(df_graph['source']).union(set(df_graph['target']))
post_handles = set(df['handle'])
print(f"Graph handles in posts:  {len(graph_handles & post_handles)} / {len(graph_handles)}")
print(f"Reply sources in posts:  {df_reply_edges['source'].isin(post_handles).sum()} / {len(df_reply_edges)}")

══════════════════════════════════════════════════
POSTS CLEAN
══════════════════════════════════════════════════
Total posts:        858
Unique URIs:        858  (should match total)
Unique authors:     602
Null text:          0  (should be 0)
Null handle:        0  (should be 0)
Null created_at:    0  (should be 0)
Date range:         2025-07-03 19:50:45+00:00 → 2026-06-23T15:19:45.212Z
User types:         {'general': 752, 'professional': 87, 'tech': 19}

══════════════════════════════════════════════════
GRAPH
══════════════════════════════════════════════════
Total edges:        10662
Edge types:         {'follows': 10219, 'reply': 443}
Null source:        0  (should be 0)
Null target:        0  (should be 0)
Unique sources:     9222
Unique targets:     294

══════════════════════════════════════════════════
PROFILES
══════════════════════════════════════════════════
Total profiles:     127
Null bios:          8
All in posts:       False  (should be True)

═════════════════════════

In [ ]:
# This is the oldest relevant post
print(df[df['created_at'] == df['created_at'].min()][['handle', 'text_clean', 'created_at']])

         handle                                         text_clean  \
236  fifisch.cz  My point is that machine translation did chang...   

                    created_at  
236  2025-07-03 19:50:45+00:00  


In [ ]:
# Check which authors do not have saved posts
missing = df_profiles[~df_profiles['handle'].isin(df['handle'])]['handle'].tolist()
print(f"Profiles not in posts: {missing}")

Profiles not in posts: ['pauseai.bsky.social', 'bot-tan.suibari.com', 'throwable.bsky.social', 'artificialbodies.net', 'spooftaca.bsky.social', 'bollabai.bsky.social', 'fliglman.bsky.social', 'aisafetymem-mir-rt.selfhosted.social', 'pauseaius.bsky.social', 'samuelxl5.bsky.social', 'sujato.bsky.social', 'gbaspgamer.bsky.social', 'puke.bsky.social', 'gungibson.bsky.social', 'xndxn.bsky.social', 'pauseainl.bsky.social', 'aisafetymemes-mirr.selfhosted.social', 'mrcheeze.github.io', 'abahc-bot.bsky.social', 'pennybee.bsky.social', 'moskov.goodventures.org']


In [ ]:
# Delete authors without posts (most likely the post was irrelevant and was filtered out)
df_profiles = df_profiles[df_profiles['handle'].isin(df['handle'])].reset_index(drop=True)
print(f"Profiles after cleanup: {len(df_profiles)}")

# re-save
df_profiles.to_csv('data/author_profiles.csv', index=False)

Profiles after cleanup: 106


In [34]:
print(f"All in posts: {df_profiles['handle'].isin(df['handle']).all()}  (should be True)")
print(f"Null bios: {df_profiles['bio'].isnull().sum()}")

All in posts: True  (should be True)
Null bios: 8


In [39]:
df_profiles.to_csv('data/author_profiles.csv', index=False)
print(f"Saved author_profiles.csv - {len(df_profiles)} profiles")
df.to_csv('data/posts_clean.csv', index=False)
print(f"Saved posts_clean.csv — {len(df)} posts")

Saved author_profiles.csv - 106 profiles
Saved posts_clean.csv — 858 posts


## Analysis

In [ ]:
# RUN FROM HERE ON SAVED DATA 
df = pd.read_csv('data/posts_clean.csv')
df_profiles = pd.read_csv('data/author_profiles.csv')
df_graph = pd.read_csv('data/graph.csv')

print(f"Posts loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nGraph DataFrame shape:", df_graph.shape)
print(f"Graph DataFrame columns:", df_graph.columns)
print(f"\nAuthor profile DataFrame shape:", df_profiles.shape)
print(f"Author profile DataFrame columns:", df_profiles.columns)

Posts loaded: 858
Columns: ['created_at', 'uri', 'cid', 'text', 'handle', 'did', 'replies', 'likes', 'reposts', 'quotes', 'hashtags', 'mentions', 'links', 'is_reply', 'reply_to', 'query', 'text_clean', 'bio', 'user_type']

Graph DataFrame shape: (10662, 3)
Graph DataFrame columns: Index(['source', 'target', 'type'], dtype='str')

Author profile DataFrame shape: (106, 6)
Author profile DataFrame columns: Index(['handle', 'display_name', 'bio', 'followers', 'following',
       'posts_count'],
      dtype='str')
